# 07 · RQ2: Which emotions are harder to recognise?

Report section 3.3 and Figure 5. The per-emotion precision, recall and F1 of
both models (Table 4) are computed in notebook 05; here they are compared
across emotions, with the clips that neither model recognised, and with
human raters.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    P = Path("/content/drive/MyDrive/Affective Project/predictions")
else:
    P = Path("../predictions")

FILES = {
    "audio_v2_test": P / "audio_v2_test_predictions.csv",
    "audio_v2_validation": P / "audio_v2_validation_predictions.csv",
    "audio_v1_validation": P / "audio_v1_validation_predictions.csv",
    "audio_v2_history": P / "audio_v2_history.csv",
    "visual_test": P / "visual_test_predictions.csv",
    "visual_validation": P / "visual_validation_predictions.csv",
}
missing = [str(p) for p in FILES.values() if not p.exists()]
assert not missing, "Missing files:\n" + "\n".join(missing)
for k, p in FILES.items():
    print(f"{k:22s} {p}")

Mounted at /content/drive
audio_v2_test          /content/drive/MyDrive/Affective Project/predictions/audio_v2_test_predictions.csv
audio_v2_validation    /content/drive/MyDrive/Affective Project/predictions/audio_v2_validation_predictions.csv
audio_v1_validation    /content/drive/MyDrive/Affective Project/predictions/audio_v1_validation_predictions.csv
audio_v2_history       /content/drive/MyDrive/Affective Project/predictions/audio_v2_history.csv
visual_test            /content/drive/MyDrive/Affective Project/predictions/visual_test_predictions.csv
visual_validation      /content/drive/MyDrive/Affective Project/predictions/visual_validation_predictions.csv


In [2]:
D = Path("/content/drive/MyDrive/Affective Project/data") if IN_COLAB else Path("../data")
DATA = {"ratings": D / "summaryTable.csv", "actors": D / "VideoDemographics.csv"}
missing = [str(p) for p in DATA.values() if not p.exists()]
assert not missing, "Missing files:\n" + "\n".join(missing)

In [3]:
CLASSES = ["Anger", "Disgust", "Fear", "Happy", "Neutral", "Sad"]
PROB = [f"prob_{c}" for c in CLASSES]


def load_split(split):
    """One row per clip: true label and both models' predictions and probabilities.
    Clips without a visual prediction are dropped (only 1076_MTI_SAD_XX in validation)."""
    v = pd.read_csv(FILES[f"visual_{split}"])
    v = v[v["status"] == "ok"]
    a = pd.read_csv(FILES[f"audio_v2_{split}"])
    a["video_id"] = a["filename"].str.replace(".wav", "", regex=False)
    m = v[["video_id", "Speaker", "true_label", "predicted_label", *PROB]].merge(
        a[["video_id", "true_label", "predicted_label", *PROB]],
        on="video_id", suffixes=("_v", "_a"), validate="one_to_one")
    assert (m["true_label_v"] == m["true_label_a"]).all(), "label mismatch"
    m = m.rename(columns={"true_label_v": "y", "predicted_label_v": "pred_v",
                          "predicted_label_a": "pred_a"}).drop(columns="true_label_a")
    return m


def ece(p, y_idx, bins=10):
    """Expected calibration error with 10 equal-width confidence bins."""
    conf, pred = p.max(1), p.argmax(1)
    correct = (pred == y_idx).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (conf > lo) & (conf <= hi)
        if s.any():
            total += s.mean() * abs(conf[s].mean() - correct[s].mean())
    return total

In [4]:
CODE = {"A": "Anger", "D": "Disgust", "F": "Fear", "H": "Happy", "N": "Neutral", "S": "Sad"}


def add_human_ratings(df):
    s = pd.read_csv(DATA["ratings"], dtype=str).rename(columns={"FileName": "video_id"})
    df = df.merge(s[["video_id", "VoiceVote", "FaceVote", "MultiModalVote"]],
                  on="video_id", how="left", validate="one_to_one")
    assert df.VoiceVote.notna().all()
    for col, new in [("VoiceVote", "h_voice"), ("FaceVote", "h_face"), ("MultiModalVote", "h_av")]:
        df[new] = df[col].map(lambda x: frozenset(CODE[c] for c in x.split(":")))
        df[new + "_correct"] = [y in s for y, s in zip(df.y, df[new])]
    df["human_disagree"] = [len(a & b) == 0 for a, b in zip(df.h_voice, df.h_face)]
    return df

In [5]:
from sklearn.metrics import f1_score, recall_score

test = load_split("test")
test["correct_v"] = test.pred_v == test.y
test["correct_a"] = test.pred_a == test.y
test = add_human_ratings(test)

## F1 and recall of each model per emotion (%)

In [6]:
per_emotion = pd.DataFrame({
    "audio_F1": 100 * f1_score(test.y, test.pred_a, labels=CLASSES, average=None),
    "audio_recall": 100 * recall_score(test.y, test.pred_a, labels=CLASSES, average=None),
    "visual_F1": 100 * f1_score(test.y, test.pred_v, labels=CLASSES, average=None),
    "visual_recall": 100 * recall_score(test.y, test.pred_v, labels=CLASSES, average=None),
}, index=CLASSES).round(1)
print(per_emotion.to_string(), "\n")
for m in ["audio", "visual"]:
    f = per_emotion[f"{m}_F1"]
    print(f"{m:6s}: hardest {f.idxmin()} ({f.min()}), easiest {f.idxmax()} ({f.max()}), "
          f"difference {f.max() - f.min():.0f} percentage points")
sad = test[test.y == "Sad"]
print(f"\nSadness clips recognised by the visual model: {sad.correct_v.sum()} of {len(sad)}")

         audio_F1  audio_recall  visual_F1  visual_recall
Anger        75.1          76.2       66.8           63.4
Disgust      61.2          55.1       79.4           82.3
Fear         64.7          59.2       61.6           66.0
Happy        68.5          65.7       95.7           95.8
Neutral      74.6          87.2       58.5           63.9
Sad          61.9          66.0       42.0           36.2 

audio : hardest Disgust (61.2), easiest Anger (75.1), difference 14 percentage points
visual: hardest Sad (42.0), easiest Happy (95.7), difference 54 percentage points

Sadness clips recognised by the visual model: 96 of 265


## Clips that neither model recognised (% of each emotion)

In [7]:
neither = 100 * (~test.correct_v & ~test.correct_a).groupby(test.y).mean()
neither.loc[CLASSES].round(1).to_frame("neither_model_correct_%")

,neither_model_correct_%
y,
Anger,9.4
Disgust,8.7
Fear,16.6
Happy,1.9
Neutral,4.8
Sad,20.0


## Comparison with human raters (Figure 5)

Recall per emotion. A human channel counts as correct when the intended
emotion is the label chosen most often by its raters.

In [8]:
recall = pd.DataFrame({
    "humans_voice_only": 100 * test.groupby("y").h_voice_correct.mean(),
    "audio_model": 100 * test.groupby("y").correct_a.mean(),
    "humans_face_only": 100 * test.groupby("y").h_face_correct.mean(),
    "visual_model": 100 * test.groupby("y").correct_v.mean(),
}).loc[CLASSES].round(1)
recall

,humans_voice_only,audio_model,humans_face_only,visual_model
y,,,,
Anger,64.9,76.2,70.9,63.4
Disgust,29.8,55.1,68.7,82.3
Fear,38.5,59.2,55.8,66.0
Happy,38.1,65.7,97.0,95.8
Neutral,97.8,87.2,98.7,63.9
Sad,25.7,66.0,30.2,36.2


In [9]:
better = recall.audio_model > recall.humans_voice_only
print("Emotions where the audio model is better than voice-only raters:", list(recall.index[better]))
print("Emotions where voice-only raters have recall below 40%:",
      list(recall.index[recall.humans_voice_only < 40]))
print("Emotions where humans are better from the face than from the voice:",
      list(recall.index[recall.humans_face_only > recall.humans_voice_only]), "(Section 4.2)")

Emotions where the audio model is better than voice-only raters: ['Anger', 'Disgust', 'Fear', 'Happy', 'Sad']
Emotions where voice-only raters have recall below 40%: ['Disgust', 'Fear', 'Happy', 'Sad']
Emotions where humans are better from the face than from the voice: ['Anger', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad'] (Section 4.2)
